# 01 · Explore the governed gold data products
**Session 3 · step 1 of 4 · ~15 min**

🧠 **The idea.** Before you query anything, learn to *see* what's governed and how it's
organized. Unity Catalog is the single place where every table, its schema, its lineage,
and its access live. An analyst who can navigate UC never has to ask "where does this
number come from?" — they can look.

🏢 **Why FHLB-Topeka cares.** Governed, discoverable data is the prerequisite the day
opens with (Session 1). This is that governance, from the analyst's seat.

In [ ]:
# ============================================================
#  WORKSHOP CONFIG (1 of 2)  —  pick your catalog, THEN run the next cell
# ============================================================
# Mode A (live, our workspace):  serverless_stable_6fhczt_catalog  (the default)
# Mode B (portable / Free Edition): type the catalog 00_LOAD_DATA used
#   (the one you created, or an existing one you loaded into). Same value in every module.
# This cell only creates the picker at the top of the notebook.
dbutils.widgets.text("catalog", "serverless_stable_6fhczt_catalog", "Catalog")
print("↑ Set the 'Catalog' widget at the top of the notebook, then run the next cell.")

In [ ]:
# ---- WORKSHOP CONFIG (2 of 2)  —  apply the selected catalog ----
CATALOG = dbutils.widgets.get("catalog").strip()
assert CATALOG, "Set the 'Catalog' widget at the top of the notebook, then re-run this cell."

GOLD   = f"{CATALOG}.fhlb_gold"     # governed, analyst-ready data products (read-only)
SILVER = f"{CATALOG}.fhlb_silver"   # cleaned/typed layer (we use the HPI time series here)

spark.sql(f"USE CATALOG {CATALOG}")
print(f"Catalog: {CATALOG}  ·  gold: {GOLD}")

In [ ]:
# ---- Verify you can read the governed gold data products ----
# You are an ANALYST here: you READ curated gold tables and build your own Genie
# space + dashboard. You do not create or drop tables. If a table is missing, tell
# your facilitator (Mode B: the 00_LOAD_DATA loader may not have finished).
expected = ["member_advance_summary", "portfolio_concentration",
            "member_collateral_capacity", "mpf_portfolio_summary",
            "housing_market_reference"]
present = {r.tableName for r in spark.sql(f"SHOW TABLES IN {GOLD}").collect()}
missing = [t for t in expected if t not in present]
if missing:
    print("⚠️  Missing gold tables:", missing)
    print("    Present:", sorted(present))
    print("    Mode A: check CATALOG in the config cell. Mode B: re-run 00_LOAD_DATA.")
else:
    print("✓ All 5 analyst gold tables are readable. You're good to go.")

## Exercise 1 — three ways to inspect a data product

You'll inspect `member_advance_summary` three ways. **Predict first:** how many columns
do you think an "advance summary" needs to answer *who owes us how much, at what rate,
maturing when*? Jot a number, then check.

**Way 1 — the Catalog UI (click).** In the left nav open **Catalog**, expand
`serverless_stable_6fhczt_catalog` → `fhlb_gold` → `member_advance_summary`. Look at the
**Columns**, **Sample Data**, **Details**, and **Lineage** tabs. Lineage shows this gold
product was built from silver — that's your provenance.

In [ ]:
# Way 2 — describe the schema from code
display(spark.sql(f"DESCRIBE TABLE {GOLD}.member_advance_summary"))

In [ ]:
%sql
-- Way 3 — peek at the data (SQL). Was your column-count prediction close?
SELECT * FROM fhlb_gold.member_advance_summary LIMIT 10

👀 **Insight.** One governed table already answers "who / how much / what rate /
maturing when" per member. That's what "gold data product" means — modeled for a question,
not a raw dump.

## Exercise 2 — what else is on the shelf?
List every gold data product, then pick one you haven't seen and describe it.

In [ ]:
%sql
SHOW TABLES IN fhlb_gold

In [ ]:
# Your turn: change the table name and run.
tbl = "member_collateral_capacity"   # <- try portfolio_concentration, mpf_portfolio_summary, ...
display(spark.sql(f"DESCRIBE TABLE {GOLD}.{tbl}"))

## 🧑‍💻 Your Turn
- Find the table that would answer **"which members are undercollateralized?"** (hint: capacity).
- Open its **Lineage** tab in the Catalog UI — what silver table feeds it?

## ⚠️ Fallback
If the Catalog UI is slow, `DESCRIBE TABLE` and `SHOW TABLES IN fhlb_gold` from code give
you the same structure in seconds.

## 🌟 Optional
Run `DESCRIBE HISTORY fhlb_gold.member_advance_summary` — every write to a governed Delta
table is versioned and auditable.

---
### ✅ Done
**Next:** open `02_sql_analysis`.